# Code-gram tokenizer & query planner — walkthrough

Phase 1 + 1.5 of [story 013](../context/stories/013-database-agnostic-code-trigram-index/spec.md). This notebook demonstrates the pure-Python primitive in `vfs.code_grams`:

- raw byte-trigram extraction from normalized UTF-8 content
- NFC + casefold for Unicode safety
- the `GramQuery` planner over the Python `sre_parse` AST (no `pg_trgm`)
- end-to-end demo: extract grams → build query → intersect candidates → authoritative regex verification

Run with `uv run jupyter lab grep_glob\ research/code_grams_walkthrough.ipynb` (or VS Code's notebook runner). No database is required.

In [1]:
import re
from collections import defaultdict
from pprint import pprint

from vfs.code_grams import (
    GramAnd,
    GramAny,
    GramOr,
    build_code_gram_query,
    fold_content,
    grams_for_fixed_string,
    iter_code_grams,
    normalize_content,
    pack_gram,
    unique_code_grams,
    unpack_gram,
)


def show_grams(grams):
    """Pretty-print a set/list of packed grams as their byte representation."""
    return sorted(unpack_gram(g) for g in grams)


def show_query(q):
    """Pretty-print a GramQuery with byte representations."""
    if isinstance(q, GramAny):
        return "ANY (no useful grams — falls back to scan)"
    if isinstance(q, GramAnd):
        return f"AND({show_grams(q.grams)})"
    if isinstance(q, GramOr):
        return "OR(\n    " + ",\n    ".join(show_query(b) for b in q.branches) + "\n)"
    return repr(q)

## 1. The tokenizer

`iter_code_grams(content)` yields every sliding 3-byte window of NFC-normalized UTF-8 bytes. Punctuation, whitespace, operators, and path separators all participate — that's what distinguishes this from `pg_trgm`.

In [2]:
for sample in [
    "foo|bar",
    "path/to/file.py",
    "async def _grep_impl(",
    "a?.b",
]:
    grams = list(iter_code_grams(sample))
    print(f"{sample!r:35s} -> {[unpack_gram(g) for g in grams]}")

'foo|bar'                           -> [b'foo', b'oo|', b'o|b', b'|ba', b'bar']
'path/to/file.py'                   -> [b'pat', b'ath', b'th/', b'h/t', b'/to', b'to/', b'o/f', b'/fi', b'fil', b'ile', b'le.', b'e.p', b'.py']
'async def _grep_impl('             -> [b'asy', b'syn', b'ync', b'nc ', b'c d', b' de', b'def', b'ef ', b'f _', b' _g', b'_gr', b'gre', b'rep', b'ep_', b'p_i', b'_im', b'imp', b'mpl', b'pl(']
'a?.b'                              -> [b'a?.', b'?.b']


### Newline normalization

`\r\n` and `\r` collapse to `\n` so an indexer doesn't store one set of grams for a CRLF file and a different set for an LF file.

In [3]:
lf = "a\nb"
crlf = "a\r\nb"
cr = "a\rb"
print("LF   bytes:", normalize_content(lf))
print("CRLF bytes:", normalize_content(crlf))
print("CR   bytes:", normalize_content(cr))
assert unique_code_grams(lf) == unique_code_grams(crlf) == unique_code_grams(cr)
print("\n✓ all three normalize to the same gram set")

LF   bytes: b'a\nb'
CRLF bytes: b'a\nb'
CR   bytes: b'a\nb'

✓ all three normalize to the same gram set


### Multi-byte UTF-8

The tokenizer walks **bytes**, not codepoints, so it sees inside multibyte UTF-8 sequences. This is critical for code that mixes ASCII and non-ASCII identifiers.

In [4]:
sample = "a✓b"
print(f"{sample!r} encodes to {sample.encode('utf-8')!r}")
for g in iter_code_grams(sample):
    print("  gram", unpack_gram(g))

'a✓b' encodes to b'a\xe2\x9c\x93b'
  gram b'a\xe2\x9c'
  gram b'\xe2\x9c\x93'
  gram b'\x9c\x93b'


### Unicode normalization (NFC)

`café` can be encoded two ways:

- **NFC** (composed): `caf\xc3\xa9` — 5 bytes
- **NFD** (decomposed): `cafe\xcc\x81` — 6 bytes

These produce *different* byte trigrams. If the index writer and query planner disagree on form, matching content is silently dropped. The library NFC-normalizes both sides:

In [5]:
nfc = "café"
nfd = "café"  # this source is NFC-encoded; show the NFD form explicitly:
import unicodedata
nfd = unicodedata.normalize("NFD", nfc)
print("NFC bytes:", nfc.encode("utf-8"))
print("NFD bytes:", nfd.encode("utf-8"))
print("NFC grams:", show_grams(unique_code_grams(nfc)))
print("NFD grams:", show_grams(unique_code_grams(nfd)))
assert unique_code_grams(nfc) == unique_code_grams(nfd)
print("\n✓ NFC normalization makes them agree")

NFC bytes: b'caf\xc3\xa9'
NFD bytes: b'cafe\xcc\x81'
NFC grams: [b'af\xc3', b'caf', b'f\xc3\xa9']
NFD grams: [b'af\xc3', b'caf', b'f\xc3\xa9']

✓ NFC normalization makes them agree


### Casefold for case-insensitive search

The folded stream is lowercased via Unicode `casefold()` (not `lower()`). German `ß` correctly folds to `ss`, so a folded query for `STRASSE` lands on indexed content containing `Straße`.

In [6]:
raw_grams = unique_code_grams("Straße")
folded_grams = unique_code_grams("Straße", folded=True)
lowercase_grams = unique_code_grams("strasse")

print("raw      :", show_grams(raw_grams))
print("folded   :", show_grams(folded_grams))
print("strasse  :", show_grams(lowercase_grams))
assert folded_grams == lowercase_grams
print("\n✓ folded(Straße) == raw(strasse)")

raw      : [b'Str', b'a\xc3\x9f', b'ra\xc3', b'tra', b'\xc3\x9fe']
folded   : [b'ass', b'ras', b'sse', b'str', b'tra']
strasse  : [b'ass', b'ras', b'sse', b'str', b'tra']

✓ folded(Straße) == raw(strasse)


## 2. The query planner

`build_code_gram_query(pattern)` returns a `GramQuery`:

- `GramAnd(grams)` — every match must contain all these grams (intersect posting lists)
- `GramOr(branches)` — every match must satisfy at least one branch (union of intersections)
- `GramAny()` — no useful grams; caller must scan

**Soundness contract:** false positives OK, false negatives FORBIDDEN. The Python regex always runs afterward as the authority.

### Tier 1 — fixed strings

In [7]:
for s in ["postgres", "foo|bar", "a?", "path/to/file.py"]:
    q = build_code_gram_query(s, fixed_strings=True)
    print(f"{s!r:25s} -> {show_query(q)}")

'postgres'                -> AND([b'gre', b'ost', b'pos', b'res', b'stg', b'tgr'])
'foo|bar'                 -> AND([b'bar', b'foo', b'oo|', b'o|b', b'|ba'])
'a?'                      -> ANY (no useful grams — falls back to scan)
'path/to/file.py'         -> AND([b'.py', b'/fi', b'/to', b'ath', b'e.p', b'fil', b'h/t', b'ile', b'le.', b'o/f', b'pat', b'th/', b'to/'])


### Tier 2 — top-level alternation becomes OR

In [9]:
for s in [
    "Table|Column",
    "foo|bar|baz",
    "foo|a.*b",        # one branch is unconstrained -> ANY (sound)
]:
    q = build_code_gram_query(s)
    print(f"{s!r}")
    print("  ->", show_query(q))
    print()

'Table|Column'
  -> OR(
    AND([b'Tab', b'abl', b'ble']),
    AND([b'Col', b'lum', b'olu', b'umn'])
)

'foo|bar|baz'
  -> OR(
    AND([b'foo']),
    AND([b'bar']),
    AND([b'baz'])
)

'foo|a.*b'
  -> ANY (no useful grams — falls back to scan)



### Tier 3 — guaranteed literal runs from regexes

In [10]:
for s in [
    "Postgres(FileSystem|Backend)",   # "Postgres" run + alternation
    "foo.*bar",                         # two literal runs separated by .*
    "[a-z]+Exception",                 # class breaks run; "Exception" survives
    r"\bfoobar\b",                    # word boundaries don't break literals
    "async def _grep_impl\\(",         # punctuation in run
]:
    q = build_code_gram_query(s)
    print(f"{s!r}")
    print("  ->", show_query(q))
    print()

'Postgres(FileSystem|Backend)'
  -> AND([b'Pos', b'gre', b'ost', b'res', b'stg', b'tgr'])

'foo.*bar'
  -> AND([b'bar', b'foo'])

'[a-z]+Exception'
  -> AND([b'Exc', b'cep', b'ept', b'ion', b'pti', b'tio', b'xce'])

'\\bfoobar\\b'
  -> AND([b'bar', b'foo', b'oba', b'oob'])

'async def _grep_impl\\('
  -> AND([b' _g', b' de', b'_gr', b'_im', b'asy', b'c d', b'def', b'ef ', b'ep_', b'f _', b'gre', b'imp', b'mpl', b'nc ', b'p_i', b'pl(', b'rep', b'syn', b'ync'])



### Tier 4 — patterns that degrade to ANY

In [11]:
for s in ["a.*b", ".*", "[a-z]{2,}", "^\\d+$"]:
    q = build_code_gram_query(s)
    print(f"{s!r:20s} -> {show_query(q)}")

'a.*b'               -> ANY (no useful grams — falls back to scan)
'.*'                 -> ANY (no useful grams — falls back to scan)
'[a-z]{2,}'          -> ANY (no useful grams — falls back to scan)
'^\\d+$'             -> ANY (no useful grams — falls back to scan)


## 3. The hard cases — Phase 1.5 fixes

The first hand-rolled tokenizer had several silent false-negative bugs surfaced by the audit. The AST-based rewrite fixes all of them. Each cell below shows the pattern, what we'd extract, and a sample matching string. The required grams must be a **subset** of the actual matched-content grams — that's the soundness check.

In [12]:
AUDIT_CASES = [
    # (pattern, sample matching content, description)
    ("(?P<name>foo)bar", "foobar", "named groups: 'name' must NOT be required"),
    ("(?i:FOO)bar", "Foobar", "scoped (?i:): in raw mode, only 'bar' is required"),
    ("foo(?#xxx)bar", "foobar", "comments: 'xxx' must NOT be required"),
    ("(?x)foo bar baz", "foobarbaz", "verbose mode: whitespace is regex syntax"),
    (r"\x66oo", "foo", r"\x escapes resolve to actual byte"),
    ("(foo)?bar", "bar", "optional group: 'foo' must NOT be required"),
    ("(abc){0,3}xyz", "xyz", "zero-or-more group: inner must NOT be required"),
    ("(?:foo){2}bar", "foofoobar", "required group: 'foo' IS required"),
    ("foo(?=bar)", "foobar", "positive lookahead: outer 'foo' is sound"),
    ("(?<!foo)bar", "xxbar", "negative lookbehind: outer 'bar' is sound"),
    (r"(foo)\1", "foofoo", "backref: dynamic content, not requirable"),
]

for pattern, sample, note in AUDIT_CASES:
    q = build_code_gram_query(pattern)
    required = q.required_grams() if not isinstance(q, GramAny) else set()
    in_content = unique_code_grams(sample)
    sound = required <= in_content
    sample_match = re.search(pattern, sample) is not None
    status = "✓" if (sound and sample_match) else "✗"
    print(f"{status} {pattern!r}  /* {note} */")
    print(f"    sample {sample!r} matches: {sample_match}")
    print(f"    query: {show_query(q)}")
    print(f"    required ⊆ content: {sound}")
    print()

✓ '(?P<name>foo)bar'  /* named groups: 'name' must NOT be required */
    sample 'foobar' matches: True
    query: AND([b'bar', b'foo'])
    required ⊆ content: True

✓ '(?i:FOO)bar'  /* scoped (?i:): in raw mode, only 'bar' is required */
    sample 'Foobar' matches: True
    query: AND([b'bar'])
    required ⊆ content: True

✓ 'foo(?#xxx)bar'  /* comments: 'xxx' must NOT be required */
    sample 'foobar' matches: True
    query: AND([b'bar', b'foo', b'oba', b'oob'])
    required ⊆ content: True

✓ '(?x)foo bar baz'  /* verbose mode: whitespace is regex syntax */
    sample 'foobarbaz' matches: True
    query: AND([b'arb', b'bar', b'baz', b'foo', b'oba', b'oob', b'rba'])
    required ⊆ content: True

✓ '\\x66oo'  /* \x escapes resolve to actual byte */
    sample 'foo' matches: True
    query: AND([b'foo'])
    required ⊆ content: True

✓ '(foo)?bar'  /* optional group: 'foo' must NOT be required */
    sample 'bar' matches: True
    query: AND([b'bar'])
    required ⊆ content: True


## 4. End-to-end: candidate generation against a mock posting index

This simulates exactly what the Postgres backend will do, but with a Python dict as the inverted index. It demonstrates the full pipeline:

1. **Index writer**: for each chunk, extract `unique_code_grams(content)` and append the chunk id to each gram's posting list.
2. **Query planner**: compile the user's regex into a `GramQuery`.
3. **Candidate selection**: union posting lists per gram, intersect across grams (or take the OR-of-intersections for `GramOr`).
4. **Authoritative match**: run Python regex on candidate chunks only — false positives are filtered out here.

In [13]:
# Mock corpus — small enough to verify by eye, varied enough to exercise the planner.
CORPUS = {
    "src/auth/login.py":
        "def login(user, password):\n    return PostgresBackend.authenticate(user, password)\n",
    "src/auth/session.py":
        "class SessionStore:\n    def __init__(self):\n        self.backend = MemoryBackend()\n",
    "src/storage/postgres_fs.py":
        "class PostgresFileSystem(DatabaseFileSystem):\n    async def grep(self, pattern):\n        ...\n",
    "src/storage/mssql_fs.py":
        "class MSSQLFileSystem(DatabaseFileSystem):\n    async def grep(self, pattern):\n        ...\n",
    "docs/readme.md":
        "# Grover\n\nGrover is the agentic filesystem.\nUse path/to/file.py for examples.\n",
    "tests/test_login.py":
        "def test_login_with_postgres(): assert PostgresBackend.authenticate('a', 'b')\n",
    "tests/test_session.py":
        "def test_session_invalidate(): SessionStore().invalidate('user-42')\n",
    "src/utils/glob.py":
        "def match_glob(path, pattern):\n    return fnmatch.fnmatchcase(path, pattern)\n",
}
print(f"corpus: {len(CORPUS)} chunks")

corpus: 8 chunks


In [14]:
# --- Index writer (raw + folded streams). Mirrors the Postgres row-store DDL
# --- planned for Phase 2: vfs_entry_chunk_grams(gram_kind, gram_key, chunk_id).
raw_index: dict[int, set[str]] = defaultdict(set)
folded_index: dict[int, set[str]] = defaultdict(set)

for chunk_id, content in CORPUS.items():
    for gram in unique_code_grams(content):
        raw_index[gram].add(chunk_id)
    for gram in unique_code_grams(content, folded=True):
        folded_index[gram].add(chunk_id)

print(f"raw index    : {len(raw_index):,} distinct grams across {len(CORPUS)} chunks")
print(f"folded index : {len(folded_index):,} distinct grams across {len(CORPUS)} chunks")

raw index    : 323 distinct grams across 8 chunks
folded index : 312 distinct grams across 8 chunks


In [15]:
def candidates_for(query, *, folded: bool) -> set[str]:
    """Resolve a GramQuery to a set of candidate chunk ids using the in-memory
    posting index. Returns the universe (all chunks) for GramAny."""
    index = folded_index if folded else raw_index
    universe = set(CORPUS.keys())
    if isinstance(query, GramAny):
        return universe
    if isinstance(query, GramAnd):
        sets = [index.get(g, set()) for g in query.grams]
        if not sets:
            return universe
        # Intersect smallest-first for efficiency.
        sets.sort(key=len)
        result = set(sets[0])
        for s in sets[1:]:
            result &= s
            if not result:
                break
        return result
    if isinstance(query, GramOr):
        result = set()
        for branch in query.branches:
            result |= candidates_for(branch, folded=folded)
        return result
    raise TypeError(query)


def grep_via_index(pattern: str, *, fixed_strings: bool = False, case_insensitive: bool = False) -> dict:
    """Full pipeline: gram candidate filter -> authoritative regex."""
    query = build_code_gram_query(pattern, fixed_strings=fixed_strings, folded=case_insensitive)
    candidates = candidates_for(query, folded=case_insensitive)

    # Authoritative match (this is what Python does after SQL narrows).
    if fixed_strings:
        compiled = re.compile(re.escape(pattern), re.IGNORECASE if case_insensitive else 0)
    else:
        compiled = re.compile(pattern, re.IGNORECASE if case_insensitive else 0)
    matches = {cid for cid in candidates if compiled.search(CORPUS[cid])}

    # Ground truth (scan everything) — for verifying soundness.
    truth = {cid for cid, content in CORPUS.items() if compiled.search(content)}

    return {
        "query": show_query(query),
        "candidate_count": len(candidates),
        "final_match_count": len(matches),
        "truth_count": len(truth),
        "matches": sorted(matches),
        "missing": sorted(truth - matches),  # MUST be empty for soundness.
        "false_positives": len(candidates - matches),
    }


# Realistic queries against the mock corpus.
QUERIES = [
    ("PostgresBackend", {"fixed_strings": True}),
    ("Postgres(FileSystem|Backend)", {}),
    ("def test_", {"fixed_strings": True}),
    ("async def grep", {"fixed_strings": True}),
    ("path/to/file.py", {"fixed_strings": True}),
    ("postgres", {"fixed_strings": True, "case_insensitive": True}),
    ("foo|bar", {}),  # no matches expected — both branches absent
    (".*", {}),         # ANY -> scan everything
]

for pattern, kwargs in QUERIES:
    print(f"--- {pattern!r}  {kwargs}")
    result = grep_via_index(pattern, **kwargs)
    pprint(result)
    assert not result["missing"], f"SOUNDNESS BUG: {result['missing']}"
    print()

--- 'PostgresBackend'  {'fixed_strings': True}
{'candidate_count': 2,
 'false_positives': 0,
 'final_match_count': 2,
 'matches': ['src/auth/login.py', 'tests/test_login.py'],
 'missing': [],
 'query': "AND([b'Bac', b'Pos', b'ack', b'cke', b'end', b'esB', b'gre', "
          "b'ken', b'ost', b'res', b'sBa', b'stg', b'tgr'])",
 'truth_count': 2}

--- 'Postgres(FileSystem|Backend)'  {}
{'candidate_count': 3,
 'false_positives': 0,
 'final_match_count': 3,
 'matches': ['src/auth/login.py',
             'src/storage/postgres_fs.py',
             'tests/test_login.py'],
 'missing': [],
 'query': "AND([b'Pos', b'gre', b'ost', b'res', b'stg', b'tgr'])",
 'truth_count': 3}

--- 'def test_'  {'fixed_strings': True}
{'candidate_count': 2,
 'false_positives': 0,
 'final_match_count': 2,
 'matches': ['tests/test_login.py', 'tests/test_session.py'],
 'missing': [],
 'query': "AND([b' te', b'def', b'ef ', b'est', b'f t', b'st_', b'tes'])",
 'truth_count': 2}

--- 'async def grep'  {'fixed_strings': 

**Read this output as:**

- `candidate_count` — chunks the gram filter shortlisted (what the SQL `WHERE gram_key IN (...) GROUP BY chunk_id HAVING COUNT(DISTINCT gram_key) = N` would return).
- `final_match_count` — chunks that actually match after Python regex verification.
- `truth_count` — what a full scan would find. **Must equal `final_match_count`** for the candidate path to be sound.
- `false_positives` — `candidate_count - final_match_count`. These are paid as Python regex CPU; the gram index trades them for far fewer SQL bytes scanned.
- `missing` — soundness violations. **Must be empty.** The final assert in the loop guards this.

Notice that `".*"` produces `ANY`, which scans the whole corpus — this is correct: there's no useful gram to narrow with.

## 5. Soundness sweep

Final paranoid check: random Python regex patterns against the corpus, comparing the candidate path to a full scan. Any pattern where `truth - matches` is non-empty is a soundness bug.

In [16]:
SWEEP = [
    "PostgresFileSystem",
    "DatabaseFileSystem",
    "async def",
    r"def \w+_login",
    "class .*FileSystem",
    "path/to/file.py",
    "Postgres|MSSQL",
    "(?i:postgres)",
    r"\bSession\w+\b",
    "# Grover",
    "foo.*bar",        # no match expected
    "a?bcd",            # quantifier handling
    "(authenticate)",  # capture group
    "path[/]to",       # char class
    r"\.py\b",        # escape + boundary
]

for pattern in SWEEP:
    for ci in (False, True):
        result = grep_via_index(pattern, case_insensitive=ci)
        flag = "-i" if ci else "  "
        sound = not result["missing"]
        status = "✓" if sound else "✗ SOUNDNESS BUG"
        print(f"{status} {flag} {pattern!r:30s} candidates={result['candidate_count']}  matches={result['final_match_count']}  truth={result['truth_count']}")
        assert sound, (pattern, ci, result)

print("\n✓ all sweep patterns sound — no false negatives")

✓    'PostgresFileSystem'           candidates=1  matches=1  truth=1
✓ -i 'PostgresFileSystem'           candidates=1  matches=1  truth=1
✓    'DatabaseFileSystem'           candidates=2  matches=2  truth=2
✓ -i 'DatabaseFileSystem'           candidates=2  matches=2  truth=2
✓    'async def'                    candidates=2  matches=2  truth=2
✓ -i 'async def'                    candidates=2  matches=2  truth=2
✓    'def \\w+_login'               candidates=1  matches=1  truth=1
✓ -i 'def \\w+_login'               candidates=1  matches=1  truth=1
✓    'class .*FileSystem'           candidates=2  matches=2  truth=2
✓ -i 'class .*FileSystem'           candidates=2  matches=2  truth=2
✓    'path/to/file.py'              candidates=1  matches=1  truth=1
✓ -i 'path/to/file.py'              candidates=1  matches=1  truth=1
✓    'Postgres|MSSQL'               candidates=4  matches=4  truth=4
✓ -i 'Postgres|MSSQL'               candidates=4  matches=4  truth=4
✓    '(?i:postgres)'              

## 6. What this enables in Phase 2

The `raw_index`/`folded_index` dicts above stand in for the upcoming Postgres `vfs_entry_chunk_grams` table:

```sql
CREATE TABLE vfs_entry_chunk_grams (
    gram_kind smallint not null,    -- 0 raw, 1 folded
    gram_key  integer  not null,    -- 24-bit packed trigram
    chunk_id  text     not null,
    PRIMARY KEY (gram_kind, gram_key, chunk_id)
);
```

The candidate query becomes:

```sql
SELECT chunk_id
FROM vfs_entry_chunk_grams
WHERE gram_kind = :kind
  AND gram_key IN (:g1, :g2, ...)
GROUP BY chunk_id
HAVING COUNT(DISTINCT gram_key) = :required_count;
```

and the existing `_collect_line_matches` Python authority runs against the candidate chunks. Same correctness story, different storage backend.